# Activity 3 — Fix hallucination

This is Activity 3 of the LLM Observability Hackathon. Run the cells top-to-bottom in a fresh kernel. The broken run is intentional; do not fix it before inspecting its trace.

## Purpose
Find a wrong baggage answer in the trace and fix the system prompt. The trace records the retrieved Gold policy, the system prompt, and the generated answer.

## What to expect
The broken prompt claims Gold members get two checked bags; the retrieved policy says one. The broken answer follows the prompt and contradicts the document. The fixed prompt follows the retrieved document and answers one. The notebook creates the traces `activity-3-hallucination-broken` and `activity-3-hallucination-fixed`.

## Bad practice
Put an assertion in the system prompt that the retrieved document does not support.

## Good practice
Ground the prompt in the retrieved document and tell the model to follow it when it conflicts with the model's assumptions.

## Why the good practice works
The generation input and the retriever output can be compared directly in the trace. When the prompt defers to the retrieved policy, the answer stays consistent with the document instead of the prompt's assertion.

## Deliverable
Both trace URLs, the contradiction you found, and the prompt change.


In [1]:
import time
from IPython.display import Markdown, display
from experiment_utils import (client, langfuse, MODEL, RUN_ID, print_summary,
                              propagate_attributes, record_trace, retrieve,
                              show_trace)


In [2]:
def run_hallucination(system_prompt, variant):
    trace_name = f"activity-3-hallucination-{variant}"
    with langfuse.start_as_current_observation(name=trace_name, as_type="span") as root:
        with propagate_attributes(
            session_id=f"activity-3-{RUN_ID}",
            tags=["activity-3", variant],
            trace_name=trace_name,
        ):
            doc = retrieve("checked bag allowance", "gold")
            with langfuse.start_as_current_observation(
                name="activity-3-retrieval",
                as_type="retriever",
                input={"query": "checked bag allowance", "filter": {"tier": "gold"}},
                output={"doc_id": doc["id"], "tier": doc["tier"], "text": doc["text"]},
            ):
                pass
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"I'm a Gold member. How many checked bags are included in my Gold fare?\n\nRetrieved policy:\n{doc['text']}"},
            ]
            started = time.time()
            response = client.chat.completions.create(
                model=MODEL, messages=messages, temperature=0, max_tokens=800,
                name="activity-3-generation",
            )
            latency = time.time() - started
            answer = response.choices[0].message.content
            trace_id = langfuse.get_current_trace_id()
            root.update(output={"answer": answer, "doc_id": doc["id"]})
    usage = response.usage
    return {
        "variant": variant,
        "answer": answer,
        "doc": doc,
        "input_tokens": getattr(usage, "prompt_tokens", None),
        "output_tokens": getattr(usage, "completion_tokens", None),
        "latency": latency,
        "trace_id": trace_id,
        "trace_url": record_trace(f"activity-3-{variant}", trace_id),
    }

broken_prompt = "You are Skyline Travel's assistant. Gold members are always entitled to two checked bags. Answer using the retrieved policy."
broken = run_hallucination(broken_prompt, "broken")
print("answer:", broken["answer"])
show_trace(broken["trace_id"])

answer: As a Gold member, you’re entitled to **two checked bags**.

The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you.
trace: http://localhost:3000/project/langfuse-observability/traces/0fef3113356c1bdcf8e696fc3656fd2c


'http://localhost:3000/project/langfuse-observability/traces/0fef3113356c1bdcf8e696fc3656fd2c'

## What went wrong?

Open the broken trace and compare the generation input with the retriever output. The retrieved Gold policy says one checked bag; the broken system prompt asserts two. Find the contradiction and fix the prompt.


In [3]:
fixed_prompt = "You are Skyline Travel's assistant. Answer using only the retrieved policy documents. If they conflict with your assumptions, follow the documents."
fixed = run_hallucination(fixed_prompt, "fixed")
print("answer:", fixed["answer"])
show_trace(fixed["trace_id"])

rows = ["| variant | retrieved doc | answer | input tokens | output tokens | latency (s) | trace |",
        "|---|---|---|---|---|---|---|"]
for run in [broken, fixed]:
    doc = run["doc"]
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['variant']} | {doc['id']} ({doc['tier']}) | {answer} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))

answer: Your Gold fare includes **one 23 kg checked bag only**. A second checked bag costs **60 EUR**.
trace: http://localhost:3000/project/langfuse-observability/traces/ea57331e1379e99a6a8d0ac069a4a08d


| variant | retrieved doc | answer | input tokens | output tokens | latency (s) | trace |
|---|---|---|---|---|---|---|
| broken | baggage-gold (gold) | As a Gold member, you’re entitled to **two checked bags**.  The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you. | 95 | 332 | 4.7 | http://localhost:3000/project/langfuse-observability/traces/0fef3113356c1bdcf8e696fc3656fd2c |
| fixed | baggage-gold (gold) | Your Gold fare includes **one 23 kg checked bag only**. A second checked bag costs **60 EUR**. | 98 | 55 | 1.0 | http://localhost:3000/project/langfuse-observability/traces/ea57331e1379e99a6a8d0ac069a4a08d |

## Wrap-up

Run this cell to print the session id and trace URLs for this activity, then flush the Langfuse client. Submit the trace URLs and your findings.


In [4]:
print_summary(3)
langfuse.flush()
print("flushed")

run id: b1bd78f1
session activity-3-b1bd78f1
activity-3-broken: http://localhost:3000/project/langfuse-observability/traces/0fef3113356c1bdcf8e696fc3656fd2c
activity-3-fixed: http://localhost:3000/project/langfuse-observability/traces/ea57331e1379e99a6a8d0ac069a4a08d
flushed
